# Step 2E — Generate Retail Sales Transactions

## 1. Project Configuration

## 2. Load Dimension Tables

## 3. Validate Dimension Tables

## 4. Generate Transaction Skeleton

## 5. Generate Transaction Dates

## 6. Assign Stores

## 7. Assign Customers

## 8. Assign Products

## 9. Generate Quantities

## 10. Generate Pricing

## 11. Generate Discounts

## 12. Calculate Financial Metrics

## 13. Assign Payment Methods

## 14. Assign Sales Channels

## 15. Build Final Fact Table

## 16. Data Quality Validation

## 17. Business Sanity Checks

## 18. Save Dataset

In [147]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

N_TRANSACTIONS = 120_000

In [148]:
N_TRANSACTIONS

120000

### Load Our Dimension Tables

In [149]:
data_path = Path("../data/raw")

df_store = pd.read_csv(
    data_path / "dim_store.csv"
)

df_product = pd.read_csv(
    data_path / "dim_product.csv"
)

df_customer = pd.read_csv(
    data_path / "dim_customer.csv"
)

df_date = pd.read_csv(
    data_path / "dim_date.csv"
)

### Check

In [150]:
df_store.shape

(25, 9)

In [151]:
df_product.shape

(500, 7)

In [152]:
df_customer.shape

(5000, 5)

In [153]:
df_date.shape

(731, 12)

### Validate Before Generating Sales

In [154]:
assert df_store["store_id"].is_unique
assert df_product["product_id"].is_unique
assert df_customer["customer_id"].is_unique
assert df_date["date"].is_unique

## 4. Generate Transaction Skeleton

In [155]:
N_TRANSACTIONS = 120_000

transaction_ids = np.arange(1, N_TRANSACTIONS + 1)

df_sales = pd.DataFrame({
    "transaction_id": [
        f"T{i:06d}" for i in transaction_ids
    ]
})

df_sales.head()

,transaction_id
0,T000001
1,T000002
2,T000003
3,T000004
4,T000005


In [156]:
df_sales.shape

(120000, 1)

In [157]:
df_sales["transaction_id"].is_unique

True

## 5. Generate Transaction Dates

In [158]:
df_date["date"] = pd.to_datetime(df_date["date"])

In [159]:
df_date[["date"]].head()

,date
0,2024-01-01
1,2024-01-02
2,2024-01-03
3,2024-01-04
4,2024-01-05


In [160]:
df_date["date"].min(), df_date["date"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

### Create Day-of-Week Behavior

In [161]:
date_weights = df_date[["date"]].copy()

date_weights["day_of_week"] = date_weights["date"].dt.dayofweek

In [162]:
weekday_weights = {
    0: 1.00,  # Monday
    1: 1.00,  # Tuesday
    2: 1.00,  # Wednesday
    3: 1.05,  # Thursday
    4: 1.20,  # Friday
    5: 1.30,  # Saturday
    6: 1.10   # Sunday
}

In [163]:
date_weights["weekday_weight"] = (
    date_weights["day_of_week"]
    .map(weekday_weights)
)

In [164]:
date_weights.head(10)

,date,day_of_week,weekday_weight
0,2024-01-01,0,1.00
1,2024-01-02,1,1.00
2,2024-01-03,2,1.00
3,2024-01-04,3,1.05
4,2024-01-05,4,1.20
5,2024-01-06,5,1.30
6,2024-01-07,6,1.10
7,2024-01-08,0,1.00
8,2024-01-09,1,1.00
9,2024-01-10,2,1.00


### Add Monthly Seasonality

In [165]:
date_weights["month"] = date_weights["date"].dt.month

In [166]:
monthly_weights = {
    1: 0.95,
    2: 0.95,
    3: 1.05,
    4: 1.00,
    5: 1.00,
    6: 1.05,
    7: 1.00,
    8: 1.00,
    9: 1.05,
    10: 1.10,
    11: 1.15,
    12: 1.20
}

In [167]:
date_weights["seasonal_weight"] = (
    date_weights["month"].map(monthly_weights)
)

### Add 2025 Growth

In [168]:
date_weights["year"] = date_weights["date"].dt.year

date_weights["year_weight"] = np.where(
    date_weights["year"] == 2025,
    1.08,
    1.00
)

### Combine the Weights

In [169]:
date_weights["weight"] = (
    date_weights["weekday_weight"]
    * date_weights["seasonal_weight"]
    * date_weights["year_weight"]
)

In [170]:
date_weights[
    [
        "date",
        "weekday_weight",
        "seasonal_weight",
        "year_weight",
        "weight"
    ]
].head(10)

,date,weekday_weight,seasonal_weight,year_weight,weight
0,2024-01-01,1.00,0.95,1.0,0.9500
1,2024-01-02,1.00,0.95,1.0,0.9500
2,2024-01-03,1.00,0.95,1.0,0.9500
3,2024-01-04,1.05,0.95,1.0,0.9975
4,2024-01-05,1.20,0.95,1.0,1.1400
5,2024-01-06,1.30,0.95,1.0,1.2350
6,2024-01-07,1.10,0.95,1.0,1.0450
7,2024-01-08,1.00,0.95,1.0,0.9500
8,2024-01-09,1.00,0.95,1.0,0.9500
9,2024-01-10,1.00,0.95,1.0,0.9500


### Convert Weights to Probabilities

In [171]:
date_weights["probability"] = (
    date_weights["weight"]
    / date_weights["weight"].sum()
)

In [172]:
date_weights["probability"].sum()

np.float64(1.0)

In [173]:
np.isclose(
    date_weights["probability"].sum(),
    1.0
)

np.True_

### Generate 120,000 Dates

In [174]:
df_sales["transaction_date"] = np.random.choice(
    date_weights["date"],
    size=N_TRANSACTIONS,
    p=date_weights["probability"]
)

In [175]:
df_sales.head()

,transaction_id,transaction_date
0,T000001,2024-10-20
1,T000002,2025-12-01
2,T000003,2025-07-03
3,T000004,2025-03-29
4,T000005,2024-05-04


### Validate the Dates

In [176]:
df_sales["transaction_date"].min()
df_sales["transaction_date"].max()

Timestamp('2025-12-31 00:00:00')

In [177]:
df_sales["transaction_date"].isna().sum()

np.int64(0)

In [178]:
df_sales.shape

(120000, 2)

### Inspect Daily Distribution

In [179]:
daily_sales_lines = (
    df_sales
    .groupby("transaction_date")
    .size()
    .reset_index(name="transaction_lines")
)

In [180]:
daily_sales_lines.head()

,transaction_date,transaction_lines
0,2024-01-01,133
1,2024-01-02,129
2,2024-01-03,131
3,2024-01-04,153
4,2024-01-05,141


In [181]:
daily_sales_lines["transaction_lines"].describe()

count    731.000000
mean     164.158687
std       23.903619
min      105.000000
25%      146.000000
50%      163.000000
75%      179.000000
max      256.000000
Name: transaction_lines, dtype: float64

### Check Monthly Distribution

In [182]:
monthly_distribution = (
    df_sales
    .assign(
        year=df_sales["transaction_date"].dt.year,
        month=df_sales["transaction_date"].dt.month
    )
    .groupby(["year", "month"])
    .size()
    .reset_index(name="transaction_lines")
)

In [183]:
monthly_distribution

,year,month,transaction_lines
0,2024,1,4365
1,2024,2,4276
2,2024,3,5012
3,2024,4,4494
4,2024,5,4702
5,2024,6,4816
6,2024,7,4642
7,2024,8,4640
8,2024,9,4942
9,2024,10,4924


### Compare 2024 vs 2025

In [184]:
year_distribution = (
    df_sales["transaction_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

year_distribution

transaction_date
2024    57675
2025    62325
Name: count, dtype: int64

### Check Day-of-Week Distribution

In [185]:
dow_distribution = (
    df_sales["transaction_date"]
    .dt.day_name()
    .value_counts()
)

In [186]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_distribution = dow_distribution.reindex(day_order)

dow_distribution

transaction_date
Monday       15808
Tuesday      15863
Wednesday    15938
Thursday     16262
Friday       18679
Saturday     20256
Sunday       17194
Name: count, dtype: int64

In [187]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_distribution = dow_distribution.reindex(day_order)

dow_distribution

transaction_date
Monday       15808
Tuesday      15863
Wednesday    15938
Thursday     16262
Friday       18679
Saturday     20256
Sunday       17194
Name: count, dtype: int64

### Assign Stores

In [188]:
df_store.head()

,store_id,store_name,city,region,store_type,store_size_sqft,opening_date,target_sales,target_margin
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,2020-02-08,13023504,0.28
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,2020-04-22,12605221,0.28
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,2021-05-17,9748688,0.30
3,GM004,GulfMart Riyadh East,Riyadh,Central,Supermarket,13013,2020-10-02,9920367,0.30
4,GM005,GulfMart Riyadh West,Riyadh,Central,Supermarket,12226,2019-12-11,8949709,0.30


In [189]:
df_store.columns.tolist()

['store_id',
 'store_name',
 'city',
 'region',
 'store_type',
 'store_size_sqft',
 'opening_date',
 'target_sales',
 'target_margin']

In [190]:
df_store[
    [
        "store_id",
        "store_name",
        "city",
        "region",
        "store_type",
        "store_size_sqft",
        "target_sales"
    ]
].head(10)

,store_id,store_name,city,region,store_type,store_size_sqft,target_sales
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,13023504
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,12605221
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,9748688
3,GM004,GulfMart Riyadh East,Riyadh,Central,Supermarket,13013,9920367
4,GM005,GulfMart Riyadh West,Riyadh,Central,Supermarket,12226,8949709
5,GM006,GulfMart Riyadh Express 1,Riyadh,Central,Express,7225,2879653
6,GM007,GulfMart Riyadh Express 2,Riyadh,Central,Express,5828,3610178
7,GM008,GulfMart Jeddah North,Jeddah,Western,Hypermarket,32909,12300269
8,GM009,GulfMart Jeddah South,Jeddah,Western,Hypermarket,30588,14431373
9,GM010,GulfMart Jeddah Central,Jeddah,Western,Supermarket,14023,6566510


In [191]:
df_store["store_type"].value_counts()

store_type
Supermarket     10
Express          7
Hypermarket      5
Neighborhood     3
Name: count, dtype: int64

In [192]:
df_store["city"].value_counts()

city
Riyadh     7
Jeddah     5
Dammam     3
Makkah     3
Khobar     2
Madinah    2
Abha       2
Tabuk      1
Name: count, dtype: int64

### Create the store-type weights

In [194]:
store_type_weights = {
    "Hypermarket": 1.50,
    "Supermarket": 1.20,
    "Express": 0.80,
    "Neighborhood": 0.70
}

### Map the weights to your stores

In [197]:
df_store["store_type_weight"] = (
    df_store["store_type"]
    .map(store_type_weights)
)

In [198]:
df_store[
    [
        "store_id",
        "store_type",
        "store_type_weight"
    ]
]

,store_id,store_type,store_type_weight
0,GM001,Hypermarket,1.5
1,GM002,Hypermarket,1.5
2,GM003,Supermarket,1.2
3,GM004,Supermarket,1.2
4,GM005,Supermarket,1.2
5,GM006,Express,0.8
6,GM007,Express,0.8
7,GM008,Hypermarket,1.5
8,GM009,Hypermarket,1.5
9,GM010,Supermarket,1.2


In [199]:
print(
    "Missing store type weights:",
    df_store["store_type_weight"].isna().sum()
)

Missing store type weights: 0


### Store Size Weight

In [203]:
df_store["size_weight"] = (
    df_store["store_size_sqft"]
    / df_store["store_size_sqft"].median()
)

In [204]:
df_store[
    [
        "store_id",
        "store_size_sqft",
        "size_weight"
    ]
].head()

,store_id,store_size_sqft,size_weight
0,GM001,30251,2.312768
1,GM002,34235,2.617355
2,GM003,13929,1.064908
3,GM004,13013,0.994878
4,GM005,12226,0.934709


### Create Store Performance Factor

In [206]:
df_store["performance_factor"] = np.random.uniform(
    0.85,
    1.20,
    size=len(df_store)
).round(3)

In [207]:
df_store[
    [
        "store_id",
        "store_name",
        "store_type",
        "store_size_sqft",
        "performance_factor"
    ]
]

,store_id,store_name,store_type,store_size_sqft,performance_factor
0,GM001,GulfMart Riyadh North,Hypermarket,30251,0.895
1,GM002,GulfMart Riyadh South,Hypermarket,34235,1.039
2,GM003,GulfMart Riyadh Central,Supermarket,13929,1.065
3,GM004,GulfMart Riyadh East,Supermarket,13013,1.028
4,GM005,GulfMart Riyadh West,Supermarket,12226,1.031
5,GM006,GulfMart Riyadh Express 1,Express,7225,0.907
6,GM007,GulfMart Riyadh Express 2,Express,5828,1.060
7,GM008,GulfMart Jeddah North,Hypermarket,32909,1.192
8,GM009,GulfMart Jeddah South,Hypermarket,30588,1.192
9,GM010,GulfMart Jeddah Central,Supermarket,14023,0.972


In [208]:
df_store["performance_factor"].describe()

count    25.000000
mean      1.020320
std       0.101259
min       0.868000
25%       0.911000
50%       1.028000
75%       1.105000
max       1.192000
Name: performance_factor, dtype: float64

In [209]:
print(
    "Missing performance factors:",
    df_store["performance_factor"].isna().sum()
)

Missing performance factors: 0


### Create the Final Store Weight

In [211]:
df_store["transaction_weight"] = (
    df_store["store_type_weight"]
    * df_store["size_weight"]
    * df_store["performance_factor"]
)

In [212]:
df_store[
    [
        "store_id",
        "store_type",
        "store_size_sqft",
        "store_type_weight",
        "size_weight",
        "performance_factor",
        "transaction_weight"
    ]
].sort_values(
    "transaction_weight",
    ascending=False
)

,store_id,store_type,store_size_sqft,store_type_weight,size_weight,performance_factor,transaction_weight
7,GM008,Hypermarket,32909,1.5,2.515979,1.192,4.498570
8,GM009,Hypermarket,30588,1.5,2.338532,1.192,4.181295
1,GM002,Hypermarket,34235,1.5,2.617355,1.039,4.079147
12,GM013,Hypermarket,39537,1.5,3.022706,0.898,4.071586
0,GM001,Hypermarket,30251,1.5,2.312768,0.895,3.104890
13,GM014,Supermarket,17891,1.2,1.367813,1.109,1.820286
22,GM023,Supermarket,16431,1.2,1.256193,1.160,1.748620
10,GM011,Supermarket,16966,1.2,1.297095,1.105,1.719948
20,GM021,Supermarket,15586,1.2,1.191590,0.978,1.398450
2,GM003,Supermarket,13929,1.2,1.064908,1.065,1.360953


### Validate transaction weights

In [213]:
print(
    "Missing transaction weights:",
    df_store["transaction_weight"].isna().sum()
)

print(
    "Negative transaction weights:",
    (df_store["transaction_weight"] < 0).sum()
)

Missing transaction weights: 0
Negative transaction weights: 0


### Convert weight to prababilities

In [214]:
df_store["store_probability"] = (
    df_store["transaction_weight"]
    / df_store["transaction_weight"].sum()
)

In [215]:
print(
    "Missing probabilities:",
    df_store["store_probability"].isna().sum()
)

print(
    "Probability total:",
    df_store["store_probability"].sum()
)

Missing probabilities: 0
Probability total: 0.9999999999999999


In [216]:
print(
    "Minimum probability:",
    df_store["store_probability"].min()
)

print(
    "Maximum probability:",
    df_store["store_probability"].max()
)

Minimum probability: 0.006540704716814694
Maximum probability: 0.11845190812993542


### Assign Stores to Transactions

In [217]:
df_sales["store_id"] = np.random.choice(
    df_store["store_id"],
    size=N_TRANSACTIONS,
    p=df_store["store_probability"]
)

In [219]:
df_sales.head()

,transaction_id,transaction_date,store_id
0,T000001,2024-10-20,GM007
1,T000002,2025-12-01,GM004
2,T000003,2025-07-03,GM013
3,T000004,2025-03-29,GM004
4,T000005,2024-05-04,GM012


### Validate Store IDs

In [220]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

### Missing Values

In [221]:
df_sales["store_id"].isna().sum()

np.int64(0)

### Store Transaction Distribution

In [222]:
store_transaction_distribution = (
    df_sales["store_id"]
    .value_counts()
    .rename_axis("store_id")
    .reset_index(name="transaction_lines")
)

In [223]:
store_transaction_distribution

,store_id,transaction_lines
0,GM008,14227
1,GM009,13223
2,GM013,12949
3,GM002,12816
4,GM001,9801
5,GM014,5698
6,GM023,5620
7,GM011,5462
8,GM021,4451
9,GM003,4325


### Compare Actual Distribution With Expected Probability

In [224]:
store_distribution_check = store_transaction_distribution.merge(
    df_store[
        [
            "store_id",
            "store_probability",
            "transaction_weight",
            "store_type",
            "store_size_sqft"
        ]
    ],
    on="store_id",
    how="left"
)

In [225]:
store_distribution_check["expected_lines"] = (
    store_distribution_check["store_probability"]
    * N_TRANSACTIONS
)

In [226]:
store_distribution_check[
    [
        "store_id",
        "store_type",
        "store_size_sqft",
        "transaction_lines",
        "expected_lines"
    ]
].sort_values(
    "transaction_lines",
    ascending=False
)

,store_id,store_type,store_size_sqft,transaction_lines,expected_lines
0,GM008,Hypermarket,32909,14227,14214.228976
1,GM009,Hypermarket,30588,13223,13211.730405
2,GM013,Hypermarket,39537,12949,12865.077758
3,GM002,Hypermarket,34235,12816,12888.970980
4,GM001,Hypermarket,30251,9801,9810.590243
5,GM014,Supermarket,17891,5698,5751.597880
6,GM023,Supermarket,16431,5620,5525.153371
7,GM011,Supermarket,16966,5462,5434.556319
8,GM021,Supermarket,15586,4451,4418.713867
9,GM003,Supermarket,13929,4325,4300.232125


Check Store Performance Potential

In [227]:
df_store[
    [
        "store_id",
        "store_name",
        "city",
        "region",
        "store_type",
        "store_size_sqft",
        "transaction_weight",
        "store_probability"
    ]
].sort_values(
    "transaction_weight",
    ascending=False
)

,store_id,store_name,city,region,store_type,store_size_sqft,transaction_weight,store_probability
7,GM008,GulfMart Jeddah North,Jeddah,Western,Hypermarket,32909,4.498570,0.118452
8,GM009,GulfMart Jeddah South,Jeddah,Western,Hypermarket,30588,4.181295,0.110098
1,GM002,GulfMart Riyadh South,Riyadh,Central,Hypermarket,34235,4.079147,0.107408
12,GM013,GulfMart Dammam North,Dammam,Eastern,Hypermarket,39537,4.071586,0.107209
0,GM001,GulfMart Riyadh North,Riyadh,Central,Hypermarket,30251,3.104890,0.081755
13,GM014,GulfMart Dammam Central,Dammam,Eastern,Supermarket,17891,1.820286,0.047930
22,GM023,GulfMart Abha Central,Abha,Southern,Supermarket,16431,1.748620,0.046043
10,GM011,GulfMart Jeddah East,Jeddah,Western,Supermarket,16966,1.719948,0.045288
20,GM021,GulfMart Madinah Central,Madinah,Western,Supermarket,15586,1.398450,0.036823
2,GM003,GulfMart Riyadh Central,Riyadh,Central,Supermarket,13929,1.360953,0.035835


In [228]:
df_sales.shape

(120000, 3)

In [229]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

In [230]:
df_sales["store_id"].isna().sum()

np.int64(0)

In [231]:
df_sales["store_id"].nunique()

25

In [232]:
store_transaction_distribution

,store_id,transaction_lines
0,GM008,14227
1,GM009,13223
2,GM013,12949
3,GM002,12816
4,GM001,9801
5,GM014,5698
6,GM023,5620
7,GM011,5462
8,GM021,4451
9,GM003,4325


## Assign Customers

In [233]:
df_customer.head()

,customer_id,gender,age,city,customer_segment
0,C00001,Male,31,Riyadh,New
1,C00002,Female,38,Dammam,Regular
2,C00003,Female,46,Khobar,Regular
3,C00004,Female,29,Jeddah,Regular
4,C00005,Male,47,Madinah,Regular


In [234]:
df_customer.columns.tolist()

['customer_id', 'gender', 'age', 'city', 'customer_segment']

In [235]:
df_customer.shape

(5000, 5)

In [236]:
df_customer["customer_segment"].value_counts()

customer_segment
Regular       2169
Occasional    1520
New            809
Premium        502
Name: count, dtype: int64

In [237]:
df_customer["city"].value_counts()

city
Riyadh     1446
Jeddah     1101
Dammam      600
Makkah      538
Khobar      412
Madinah     411
Abha        293
Tabuk       199
Name: count, dtype: int64

In [238]:
df_customer["customer_id"].is_unique

True

### Create Segment Weights

In [239]:
customer_segment_weights = {
    "Premium": 3.0,
    "Regular": 1.8,
    "Occasional": 0.8,
    "New": 0.5
}

In [240]:
df_customer["segment_weight"] = (
    df_customer["customer_segment"]
    .map(customer_segment_weights)
)

In [241]:
df_customer[
    [
        "customer_id",
        "customer_segment",
        "segment_weight"
    ]
].head(10)

,customer_id,customer_segment,segment_weight
0,C00001,New,0.5
1,C00002,Regular,1.8
2,C00003,Regular,1.8
3,C00004,Regular,1.8
4,C00005,Regular,1.8
5,C00006,Occasional,0.8
6,C00007,New,0.5
7,C00008,New,0.5
8,C00009,New,0.5
9,C00010,Regular,1.8


### Add Customer-Level Variation

In [242]:
customer_variation = np.random.lognormal(
    mean=0,
    sigma=0.25,
    size=len(df_customer)
)

In [243]:
df_customer["customer_variation"] = customer_variation

In [244]:
df_customer["purchase_weight"] = (
    df_customer["segment_weight"]
    * df_customer["customer_variation"]
)

In [245]:
df_customer[
    [
        "customer_id",
        "customer_segment",
        "segment_weight",
        "customer_variation",
        "purchase_weight"
    ]
].sort_values(
    "purchase_weight",
    ascending=False
).head(10)

,customer_id,customer_segment,segment_weight,customer_variation,purchase_weight
3212,C03213,Premium,3.0,2.162457,6.487372
1218,C01219,Premium,3.0,2.130256,6.390769
390,C00391,Premium,3.0,1.994325,5.982974
1778,C01779,Premium,3.0,1.969607,5.908822
3989,C03990,Premium,3.0,1.935143,5.805430
4261,C04262,Premium,3.0,1.853739,5.561216
4151,C04152,Premium,3.0,1.847214,5.541641
209,C00210,Premium,3.0,1.789030,5.367091
2100,C02101,Premium,3.0,1.778390,5.335170
1880,C01881,Premium,3.0,1.746607,5.239821


### Geographic Affinity

### Build a Store-City Lookup

In [246]:
store_city_map = (
    df_store[
        ["store_id", "city"]
    ]
    .drop_duplicates()
)

In [247]:
store_city_map.head()

,store_id,city
0,GM001,Riyadh
1,GM002,Riyadh
2,GM003,Riyadh
3,GM004,Riyadh
4,GM005,Riyadh


### Assign Customers by Geographic Pool

### Create a Store Region Weight

In [249]:
df_store[
    ["store_id", "city", "region"]
].head(20)

,store_id,city,region
0,GM001,Riyadh,Central
1,GM002,Riyadh,Central
2,GM003,Riyadh,Central
3,GM004,Riyadh,Central
4,GM005,Riyadh,Central
5,GM006,Riyadh,Central
6,GM007,Riyadh,Central
7,GM008,Jeddah,Western
8,GM009,Jeddah,Western
9,GM010,Jeddah,Western


In [250]:
set(df_customer["city"]) - set(df_store["city"])

set()

In [251]:
set(df_store["city"]) - set(df_customer["city"])

set()

### Prepare Customer Pools

In [252]:
customer_pools = {
    city: group["customer_id"].to_numpy()
    for city, group in df_customer.groupby("city")
}

In [253]:
customer_weight_pools = {
    city: group["purchase_weight"].to_numpy()
    for city, group in df_customer.groupby("city")
}

### Create a Customer Assignment Array

In [254]:
df_sales["customer_id"] = None

In [255]:
store_city_lookup = (
    df_store
    .set_index("store_id")["city"]
    .to_dict()
)

In [256]:
df_sales["store_city"] = (
    df_sales["store_id"]
    .map(store_city_lookup)
)

In [257]:
df_sales[
    ["transaction_id", "store_id", "store_city"]
].head()

,transaction_id,store_id,store_city
0,T000001,GM007,Riyadh
1,T000002,GM004,Riyadh
2,T000003,GM013,Dammam
3,T000004,GM004,Riyadh
4,T000005,GM012,Jeddah


### Assign Customers by City

In [258]:
for city in df_sales["store_city"].dropna().unique():

    mask = df_sales["store_city"] == city
    n = mask.sum()

    customers = customer_pools[city]
    weights = customer_weight_pools[city]

    probabilities = weights / weights.sum()

    df_sales.loc[mask, "customer_id"] = np.random.choice(
        customers,
        size=n,
        p=probabilities
    )

### Validate Customer Assignment

In [259]:
df_sales["customer_id"].isna().sum()

np.int64(0)

In [260]:
df_sales["customer_id"].isin(
    df_customer["customer_id"]
).all()

np.True_

In [261]:
df_sales["customer_id"].nunique()

4982

### Check Customer Purchase Frequency

In [262]:
customer_frequency = (
    df_sales["customer_id"]
    .value_counts()
    .rename_axis("customer_id")
    .reset_index(name="transaction_lines")
)

In [263]:
customer_frequency["transaction_lines"].describe()

count    4982.000000
mean       24.086712
std        18.782938
min         1.000000
25%        10.000000
50%        19.000000
75%        34.000000
max       159.000000
Name: transaction_lines, dtype: float64

### Compare Frequency by Segment

In [264]:
customer_frequency = customer_frequency.merge(
    df_customer[
        [
            "customer_id",
            "customer_segment"
        ]
    ],
    on="customer_id",
    how="left"
)

In [265]:
customer_frequency.groupby(
    "customer_segment"
)["transaction_lines"].describe()

,count,mean,std,min,25%,50%,75%,max
customer_segment,,,,,,,,
New,795.0,8.427673,4.812948,1.0,5.0,8.0,12.0,25.0
Occasional,1516.0,13.498021,7.039399,1.0,8.0,13.0,18.0,47.0
Premium,502.0,52.109562,25.277343,2.0,34.0,51.5,69.0,159.0
Regular,2169.0,30.741355,15.011341,1.0,19.0,29.0,41.0,101.0


### Check Geographic Affinity

In [266]:
customer_store_city_check = df_sales.merge(
    df_customer[
        ["customer_id", "city"]
    ],
    on="customer_id",
    how="left"
)

In [267]:
(
    customer_store_city_check["store_city"]
    ==
    customer_store_city_check["city"]
).mean()

np.float64(1.0)

In [272]:
df_sales.head()

,transaction_id,transaction_date,store_id,customer_id
0,T000001,2024-10-20,GM007,C03392
1,T000002,2025-12-01,GM004,C02634
2,T000003,2025-07-03,GM013,C01832
3,T000004,2025-03-29,GM004,C02598
4,T000005,2024-05-04,GM012,C01959


## Assigns Products

### Category Demand Weight

In [313]:
category_weights = {
    "Food & Beverages": 1.30,
    "Fresh Food": 1.25,
    "Health & Wellness": 1.05,
    "Personal Care": 0.95,
    "Beauty": 0.90,
    "Household": 0.90,
    "Baby Care": 0.85,
    "Electronics": 0.75
}

In [314]:
df_product["category"].unique()

<StringArray>
[        'Baby Care',            'Beauty',     'Personal Care',
       'Electronics',  'Food & Beverages',        'Fresh Food',
 'Health & Wellness',         'Household']
Length: 8, dtype: str

### Create the Category Weight

In [315]:
df_product["category_weight"] = (
    df_product["category"]
    .map(category_weights)
)

In [316]:
print(
    "Missing category weights:",
    df_product["category_weight"].isna().sum()
)

Missing category weights: 0


In [318]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "category_weight"
    ]
].head(20)

,product_id,product_name,category,category_weight
0,P0001,Northern Star Baby Care Item 001,Baby Care,0.85
1,P0002,Rimal Beauty Item 002,Beauty,0.90
2,P0003,Nakhla Personal Care Item 003,Personal Care,0.95
3,P0004,Rimal Electronics Item 004,Electronics,0.75
4,P0005,Arabian Select Baby Care Item 005,Baby Care,0.85
5,P0006,Gulf Fresh Food & Beverages Item 006,Food & Beverages,1.30
6,P0007,Sahara Home Fresh Food Item 007,Fresh Food,1.25
7,P0008,Arabian Select Electronics Item 008,Electronics,0.75
8,P0009,Golden Basket Fresh Food Item 009,Fresh Food,1.25
9,P0010,Golden Basket Baby Care Item 010,Baby Care,0.85


In [319]:
df_product.groupby(
    "category"
)["category_weight"].agg(
    ["count", "min", "max"]
)

,count,min,max
category,,,
Baby Care,34,0.85,0.85
Beauty,49,0.90,0.90
Electronics,42,0.75,0.75
Food & Beverages,122,1.30,1.30
Fresh Food,70,1.25,1.25
Health & Wellness,56,1.05,1.05
Household,71,0.90,0.90
Personal Care,56,0.95,0.95


In [320]:
df_product["category"].unique()

<StringArray>
[        'Baby Care',            'Beauty',     'Personal Care',
       'Electronics',  'Food & Beverages',        'Fresh Food',
 'Health & Wellness',         'Household']
Length: 8, dtype: str

In [321]:
df_product["category_weight"].isna().sum()

np.int64(0)

### Product-Level Popularity

In [322]:
product_variation = np.random.lognormal(
    mean=0,
    sigma=0.45,
    size=len(df_product)
)

In [323]:
df_product["product_variation"] = product_variation

### Final product demand weight

In [324]:
df_product["product_demand_weight"] = (
    df_product["category_weight"]
    * df_product["product_variation"]
)

### Inspect the highest-demand products

In [325]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "product_demand_weight"
    ]
].sort_values(
    "product_demand_weight",
    ascending=False
).head(15)

,product_id,product_name,category,product_demand_weight
94,P0095,Nakhla Fresh Food Item 095,Fresh Food,3.152404
350,P0351,Desert Harvest Food & Beverages Item 351,Food & Beverages,3.031932
295,P0296,Nakhla Fresh Food Item 296,Fresh Food,2.912448
167,P0168,Desert Harvest Food & Beverages Item 168,Food & Beverages,2.665747
164,P0165,Palm Valley Food & Beverages Item 165,Food & Beverages,2.632034
65,P0066,Gulf Fresh Health & Wellness Item 066,Health & Wellness,2.608152
230,P0231,Arabian Select Food & Beverages Item 231,Food & Beverages,2.561926
352,P0353,Sahara Home Food & Beverages Item 353,Food & Beverages,2.546637
79,P0080,Oasis Choice Personal Care Item 080,Personal Care,2.487365
144,P0145,Golden Basket Food & Beverages Item 145,Food & Beverages,2.370468


In [326]:
df_product[
    [
        "product_id",
        "product_name",
        "category",
        "product_demand_weight"
    ]
].sort_values(
    "product_demand_weight"
).head(15)

,product_id,product_name,category,product_demand_weight
107,P0108,Oasis Choice Household Item 108,Household,0.269272
453,P0454,Al Waha Fresh Food Item 454,Fresh Food,0.345890
423,P0424,Al Waha Electronics Item 424,Electronics,0.376620
224,P0225,Al Waha Health & Wellness Item 225,Health & Wellness,0.377257
274,P0275,Golden Basket Electronics Item 275,Electronics,0.382586
200,P0201,Gulf Fresh Beauty Item 201,Beauty,0.383031
376,P0377,Rimal Baby Care Item 377,Baby Care,0.385223
28,P0029,Arabian Select Food & Beverages Item 029,Food & Beverages,0.397090
71,P0072,Al Waha Electronics Item 072,Electronics,0.398409
357,P0358,Northern Star Beauty Item 358,Beauty,0.401859


### Create Store Assortment Probability

In [337]:
assortment_probability = {
    "Hypermarket": 0.90,
    "Supermarket": 0.75,
    "Express": 0.45,
    "Neighborhood": 0.35
}

In [338]:
df_store["assortment_probability"] = (
    df_store["store_type"]
    .map(assortment_probability)
)

In [340]:
df_store[
    [
        "store_id",
        "store_type",
        "assortment_probability"
    ]
]

,store_id,store_type,assortment_probability
0,GM001,Hypermarket,0.90
1,GM002,Hypermarket,0.90
2,GM003,Supermarket,0.75
3,GM004,Supermarket,0.75
4,GM005,Supermarket,0.75
5,GM006,Express,0.45
6,GM007,Express,0.45
7,GM008,Hypermarket,0.90
8,GM009,Hypermarket,0.90
9,GM010,Supermarket,0.75


### Store-Level Product Assignment

In [341]:
product_ids = df_product["product_id"].to_numpy()

product_weights = (
    df_product["product_demand_weight"].to_numpy()
)

In [342]:
product_probabilities = (
    product_weights / product_weights.sum()
)

### Create Product Assignment Array

In [343]:
df_sales["product_id"] = None

### Build Store Product Pools

In [344]:
store_product_pools = {}

for _, store in df_store.iterrows():

    store_id = store["store_id"]
    store_type = store["store_type"]

    probability = assortment_probability[store_type]

    mask = np.random.random(len(df_product)) < probability

    eligible_products = df_product.loc[mask]

    # Safety check
    if len(eligible_products) == 0:
        eligible_products = df_product

    store_product_pools[store_id] = eligible_products

### Inspect Store Assortment

In [349]:
store_assortment_summary = pd.DataFrame({
    "store_id": store_product_pools.keys(),
    "assortment_size": [
        len(products)
        for products in store_product_pools.values()
    ]
})

In [350]:
store_assortment_summary

,store_id,assortment_size
0,GM001,454
1,GM002,452
2,GM003,374
3,GM004,369
4,GM005,365
5,GM006,221
6,GM007,242
7,GM008,449
8,GM009,448
9,GM010,383


### Assign Products to Transactions

In [351]:
for store_id in df_sales["store_id"].unique():

    mask = df_sales["store_id"] == store_id
    n = mask.sum()

    product_pool = store_product_pools[store_id]

    weights = product_pool["product_demand_weight"].to_numpy()

    probabilities = weights / weights.sum()

    df_sales.loc[mask, "product_id"] = np.random.choice(
        product_pool["product_id"].to_numpy(),
        size=n,
        p=probabilities
    )

### Validate Product IDs

In [352]:
df_sales["product_id"].isna().sum()

np.int64(0)

In [353]:
df_sales["product_id"].isin(
    df_product["product_id"]
).all()

np.True_

In [354]:
df_sales["product_id"].nunique()

500

### Check Product Distribution

In [355]:
product_sales_distribution = (
    df_sales["product_id"]
    .value_counts()
    .rename_axis("product_id")
    .reset_index(name="transaction_lines")
)

In [356]:
product_sales_distribution.head(20)

,product_id,transaction_lines
0,P0296,618
1,P0095,609
2,P0351,606
3,P0145,589
4,P0168,579
5,P0290,551
6,P0412,535
7,P0231,535
8,P0066,533
9,P0413,510


### Category Distribution

In [357]:
product_category_check = product_sales_distribution.merge(
    df_product[
        [
            "product_id",
            "category"
        ]
    ],
    on="product_id",
    how="left"
)

In [358]:
category_distribution = (
    product_category_check
    .groupby("category")["transaction_lines"]
    .sum()
    .sort_values(ascending=False)
)

category_distribution

category
Food & Beverages     37161
Fresh Food           20194
Household            13782
Health & Wellness    12599
Personal Care        12513
Beauty                9293
Electronics           8076
Baby Care             6382
Name: transaction_lines, dtype: int64

### Bring Product Price Into df_sales

In [359]:
product_price_map = (
    df_product
    .set_index("product_id")["selling_price"]
    .to_dict()
)

In [360]:
df_sales["unit_price_reference"] = (
    df_sales["product_id"]
    .map(product_price_map)
)

In [361]:
df_sales[
    [
        "product_id",
        "unit_price_reference"
    ]
].head()

,product_id,unit_price_reference
0,P0014,40.85
1,P0485,70.79
2,P0340,52.63
3,P0058,6.33
4,P0073,29.95


In [362]:
df_sales["unit_price_reference"].isna().sum()

np.int64(0)

### Generate Quantity

In [363]:
price_median = df_product["selling_price"].median()

In [364]:
quantity_factor = (
    price_median
    / df_sales["unit_price_reference"]
)

In [365]:
quantity_factor = quantity_factor.clip(
    lower=0.5,
    upper=3.0
)

In [366]:
base_quantity = np.random.poisson(
    lam=1.2,
    size=len(df_sales)
) + 1

In [367]:
df_sales["quantity"] = np.round(
    base_quantity * quantity_factor
).astype(int)

In [368]:
df_sales["quantity"] = (
    df_sales["quantity"]
    .clip(lower=1, upper=10)
)

### Inspect Quantity

In [369]:
df_sales["quantity"].describe()

count    120000.000000
mean          3.025883
std           2.328317
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          10.000000
Name: quantity, dtype: float64

In [370]:
df_sales["quantity"].value_counts().sort_index()

quantity
1     37426
2     29274
3     19552
4      8652
5      5441
6      9462
7      1886
8       892
9      4597
10     2818
Name: count, dtype: int64

### Quantity by Category

In [371]:
quantity_category_check = df_sales.merge(
    df_product[
        [
            "product_id",
            "category"
        ]
    ],
    on="product_id",
    how="left"
)

In [372]:
quantity_category_check.groupby(
    "category"
)["quantity"].mean().sort_values(
    ascending=False
)

category
Food & Beverages     4.063427
Fresh Food           3.271219
Personal Care        3.038920
Household            2.995066
Baby Care            2.283767
Health & Wellness    1.853957
Beauty               1.765092
Electronics          1.536157
Name: quantity, dtype: float64

In [373]:
df_sales.drop(
    columns="unit_price_reference",
    inplace=True
)

In [375]:
df_sales.columns.tolist()

['transaction_id',
 'transaction_date',
 'store_id',
 'customer_id',
 'product_id',
 'quantity']

In [376]:
df_sales.shape

(120000, 6)

## Critical Validation

### Transaction IDs

In [377]:
df_sales["transaction_id"].is_unique

True

In [378]:
df_sales["store_id"].isin(
    df_store["store_id"]
).all()

np.True_

In [379]:
df_sales["customer_id"].isin(
    df_customer["customer_id"]
).all()

np.True_

In [380]:
df_sales["product_id"].isin(
    df_product["product_id"]
).all()

np.True_

In [381]:
(df_sales["quantity"] > 0).all()

np.True_

In [382]:
df_sales["product_id"].isna().sum()

np.int64(0)

In [383]:
df_sales["quantity"].isna().sum()

np.int64(0)